In [ ]:
# Install and Setup
!pip install openai -q

import os
from openai import OpenAI
import json

# Paste your key here, or use Colab Secrets (recommended)
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("myopenaIKey")

# os.environ["OPENAI_API_KEY"] = "sk-your-key-here"
client = OpenAI()
print("Client ready.")

Client ready.


In [ ]:
# Define the Tools
import random
import requests
import json
from datetime import datetime

def get_vehicle_state():
    """Returns current battery state of the EV."""
    return {"soc_pct": 40, "plugged_in": True,
            "battery_kwh": 75, "target_soc_pct": 80}
def get_electricity_prices(args: dict) -> dict:
    """
    Fetch electricity prices from Elpriset API
    args should contain:
    - date: "YYYY-MM-DD" (optional, defaults to today)
    - region: "SE1", "SE2", "SE3", or "SE4" (optional, defaults to "SE3" for Stockholm)
    """
    date = args.get("date", datetime.now().strftime("%Y-%m-%d"))
    region = args.get("region", "SE3")  # SE3 = Stockholm (your location)

    # Parse the date
    try:
        dt = datetime.strptime(date, "%Y-%m-%d")
        year = dt.strftime("%Y")
        month = dt.strftime("%m")
        day = dt.strftime("%d")
    except ValueError:
        return {"error": f"Invalid date format. Use YYYY-MM-DD"}

    # Build the API URL
    url = f"https://www.elprisetjustnu.se/api/v1/prices/{year}/{month}-{day}_{region}.json"

    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        prices = response.json()

        # Format the response nicely
        return {
            "date": date,
            "region": region,
            "prices": prices,
            "average_sek": round(sum(p["SEK_per_kWh"] for p in prices) / len(prices), 2),
            "min_price": round(min(p["SEK_per_kWh"] for p in prices), 2),
            "max_price": round(max(p["SEK_per_kWh"] for p in prices), 2)
        }
    except requests.exceptions.RequestException as e:
        return {"error": f"Failed to fetch prices: {str(e)}"}

#The below function is a mock function created initially to store price per hour. Ignore this
#def get_grid_tariff(hours: int):
#    """Returns simulated electricity prices for next N hours."""
#    base = [2.1, 1.8, 1.2, 0.8, 0.7, 0.9,
#            1.4, 2.5, 3.2, 3.5, 3.1, 2.8,
#            2.4, 2.2, 2.0, 2.1, 2.9, 3.4,
#            3.6, 3.8, 3.2, 2.6, 2.3, 2.0]
#    return [{"hour": i, "price_sek_per_kwh": base[i % 24]} for i in range(hours)]

def get_user_schedule():
    """Returns when the user needs the car next."""
    return {"next_departure": "07:00",
            "hours_until_departure": 7,
            "flexible": False}

def set_charging_schedule(start_hour: int, end_hour: int, rate_kw: float):
    """Sends charging plan to EVSE (simulated)."""
    energy = rate_kw * (end_hour - start_hour)
    cost = energy * 0.85
    return {"confirmed": True, "start_hour": start_hour,
            "end_hour": end_hour, "rate_kw": rate_kw,
            "estimated_cost_sek": round(cost, 2)}

In [ ]:
#Tool definitions (tells the LLM what tools exist)
tools = [
  {"type": "function", "function": {
    "name": "get_vehicle_state",
    "description": "Get current EV battery state: SoC, plug status, capacity.",
    "parameters": {"type": "object", "properties": {}, "required": []}}},

  #{"type": "function", "function": {
  #  "name": "get_grid_tariff",
  # "description": "Get electricity prices (SEK/kWh) for the next N hours.",
  #  "parameters": {"type": "object",
  #    "properties": {"hours": {"type": "integer",
  #      "description": "Number of hours ahead to fetch"}},
  #   "required": ["hours"]}}},

 { "type": "function",
        "function": {
            "name": "get_electricity_prices",
            "description": "Fetch Swedish electricity prices (spotpris) from Elpriset API. Returns hourly prices for a specific date and region in SEK per kWh.",
            "parameters": {
                "type": "object",
                "properties": {
                    "date": {
                        "type": "string",
                        "description": "Date in YYYY-MM-DD format (e.g., 2026-05-11). Defaults to today."
                    },
                    "region": {
                        "type": "string",
                        "enum": ["SE1", "SE2", "SE3", "SE4"],
                        "description": "Swedish price region: SE1=Luleå, SE2=Sundsvall, SE3=Stockholm, SE4=Malmö. Defaults to SE3."
                    }
                },
                "required": []
            }
        }
 },

  {"type": "function", "function": {
    "name": "get_user_schedule",
    "description": "Get user's next departure time and schedule flexibility.",
    "parameters": {"type": "object", "properties": {}, "required": []}}},

  {"type": "function", "function": {
    "name": "set_charging_schedule",
    "description": "Send the final charging plan to the EVSE.",
    "parameters": {"type": "object",
      "properties": {
        "start_hour": {"type": "integer", "description": "Hour to start (0-23)"},
        "end_hour":   {"type": "integer", "description": "Hour to stop (0-23)"},
        "rate_kw":    {"type": "number",  "description": "Charge rate in kW (max 11)"}},
      "required": ["start_hour", "end_hour", "rate_kw"]}}}
]

In [ ]:
# The Agent Loop:
TOOL_MAP = {
    "get_vehicle_state":    lambda _: get_vehicle_state(),
    #"get_grid_tariff":      lambda a: get_grid_tariff(**a),
    "get_electricity_prices" : get_electricity_prices,
    "get_user_schedule":    lambda _: get_user_schedule(),
    "set_charging_schedule":lambda a: set_charging_schedule(**a),
}

SYSTEM_PROMPT = """You are an EV charging optimizer agent.
Your job: decide the optimal charging schedule given vehicle state,
electricity prices, and the user's departure time.
Rules: never let SoC drop below 20%. Max charger rate is 11 kW.
Always call set_charging_schedule to confirm the plan.
Explain your decision in plain language at the end."""

def run_agent(user_request: str):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_request}
    ]

    print(f"\n--- Agent starting ---")
    for step in range(10):
        response = client.chat.completions.create(
            model="gpt-5.5",
            messages=messages,
            tools=tools
        )
        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append(msg)
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"  Tool call: {tc.function.name}({args})")
                result = TOOL_MAP[tc.function.name](args)
                print(f"  Result:    {result}")
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result)
                })
        else:
            print(f"\n--- Agent decision ---\n{msg.content}")
            return msg.content

    return "Agent did not finish within 10 steps."

The agent loop in Cell 4 is the heart of everything — it keeps calling the LLM until there are no more t**ool_calls** in the response. That's the ReAct pattern: reason, act, observe result, repeat.
The TOOL_MAP dictionary is how tool names (strings from the LLM) get dispatched to real Python functions. This is the bridge between the LLM world and your code.
The three commented-out scenarios in Cell 5 are your first tests — try all three and verify the agent behaves sensibly for each. The "15% battery, leaving in 1 hour" case in particular should trigger immediate charging regardless of tariff prices.

In [ ]:
# Try different scenarios:
#run_agent("My EV is plugged in. Optimize my charging for tonight.")

run_agent("I need to leave by 6am tomorrow from Luleå. Charge as cheaply as possible.")
# run_agent("Battery is at 15%. I need to leave in 1 hour — charge now.")


--- Agent starting ---
  Tool call: get_vehicle_state({})
  Result:    {'soc_pct': 40, 'plugged_in': True, 'battery_kwh': 75, 'target_soc_pct': 80}
  Tool call: get_user_schedule({})
  Result:    {'next_departure': '07:00', 'hours_until_departure': 7, 'flexible': False}
  Tool call: get_electricity_prices({'date': '2026-05-11', 'region': 'SE1'})
  Result:    {'date': '2026-05-11', 'region': 'SE1', 'prices': [{'SEK_per_kWh': 0.87415, 'EUR_per_kWh': 0.08044, 'EXR': 10.86706, 'time_start': '2026-05-11T00:00:00+02:00', 'time_end': '2026-05-11T00:15:00+02:00'}, {'SEK_per_kWh': 0.81166, 'EUR_per_kWh': 0.07469, 'EXR': 10.86706, 'time_start': '2026-05-11T00:15:00+02:00', 'time_end': '2026-05-11T00:30:00+02:00'}, {'SEK_per_kWh': 0.76624, 'EUR_per_kWh': 0.07051, 'EXR': 10.86706, 'time_start': '2026-05-11T00:30:00+02:00', 'time_end': '2026-05-11T00:45:00+02:00'}, {'SEK_per_kWh': 0.75113, 'EUR_per_kWh': 0.06912, 'EXR': 10.86706, 'time_start': '2026-05-11T00:45:00+02:00', 'time_end': '2026-05-11T0

'Confirmed: charging is scheduled from **01:00 to 04:00** at **10 kW**.\n\nWhy this plan:\n- You’re currently at **40% SoC** with a **75 kWh** battery.\n- Target is **80%**, so you need about **30 kWh**.\n- Charging for **3 hours at 10 kW** adds about **30 kWh**.\n- This avoids the more expensive early-morning prices closer to **06:00** and uses the cheapest overnight window available with the charger schedule.\n- Your SoC stays safely above the **20% minimum** the whole time.\n\nEstimated electricity cost: **about 25.5 SEK**.'